# Notebook B - Concept-grounded policy + faithfulness reward (H1, multi-class)

Reads the frozen cache from Notebook A and trains a small **concept -> diagnosis** policy with single-step policy gradient (REINFORCE). Diagnosis is a **non-linear (MLP) function of the stated concepts only**, over the **5 Derm7pt diagnosis groups** (NEV, MEL, BCC, SK, MISC). Because the diagnosis sees only the concepts, the explanation is causal by construction; the open question is whether the head reasons *clinically*.

**Hypothesis H1:** adding a faithfulness reward (the melanoma decision must agree with the 7-point-checklist verdict computed from the stated concepts) raises faithfulness -- both **rule-consistency under intervention** and **monotonicity** of P(melanoma) in the 7-point criteria -- without lowering diagnostic performance, vs. a correctness-only reward.

Why multi-class + non-linear head: with a linear binary head the 7-point rule is trivially representable, so faithfulness saturates at 1.0. The multi-class non-linear setting lets correctness and clinical faithfulness genuinely compete, so the result is meaningful.

Concept layer = supervised probe on the ground-truth concept labels in the cache (zero-shot prompts were weak for 2-3 criteria). Runs on cached vectors; CPU is fine.

**Input:** attach the dataset created from Notebook A's output (must contain features.npz and meta.parquet).

In [ ]:
# --- 1. Load cache + 5-class labels ---
import os, glob, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

CACHE_DIR = ''   # leave empty to auto-search
roots = [CACHE_DIR] if CACHE_DIR else ['/kaggle/input', '.', '..']

def find_one(name, roots):
    for r in roots:
        if not r:
            continue
        hits = sorted(glob.glob(os.path.join(r, '**', name), recursive=True))
        if hits:
            return hits[0]
    return None

fp = find_one('features.npz', roots)
if fp is None:
    raise FileNotFoundError('features.npz not found. Attach the Notebook A output dataset.')
print('cache:', fp)
d = np.load(fp, allow_pickle=True)
mp = find_one('meta.parquet', [os.path.dirname(fp)] + roots)
meta = pd.read_parquet(mp)
assert len(meta) == len(d['emb_derm']), 'meta/cache row mismatch'

torch.manual_seed(0)
split = d['split'].astype('U8')
X = torch.tensor(d['emb_derm'], dtype=torch.float32)
C = torch.tensor(d['gt_concepts'], dtype=torch.float32)
W7 = torch.tensor([2., 2., 2., 1., 1., 1., 1.])
CONCEPTS = ['atyp_pigment_net', 'blue_white_veil', 'atyp_vascular', 'irreg_streaks', 'irreg_pigment', 'irreg_dots', 'regression']

# 5-group Derm7pt diagnosis taxonomy (substring map -> robust to spelling variants)
CLASSES = ['NEV', 'MEL', 'BCC', 'SK', 'MISC']
MEL_IDX = CLASSES.index('MEL')

def to5(s):
    s = str(s).lower()
    if 'melanoma' in s:
        return 'MEL'
    if 'nevus' in s:
        return 'NEV'
    if 'basal cell' in s:
        return 'BCC'
    if 'seborrheic' in s:
        return 'SK'
    return 'MISC'

y5 = torch.tensor([CLASSES.index(to5(s)) for s in meta['diagnosis']], dtype=torch.long)

tr = split == 'train'; te = split == 'test'
Xtr, Ctr, ytr = X[tr], C[tr], y5[tr]
Xte, Cte, yte = X[te], C[te], y5[te]
print('train/test:', int(tr.sum()), int(te.sum()), '| classes:', CLASSES)
print('train class counts:', [int((ytr == k).sum()) for k in range(5)])

In [ ]:
# --- 2. Metrics & rule ---
def auc(y_true, score):
    y_true = np.asarray(y_true).astype(float)
    score = np.asarray(score).astype(float)
    pos = score[y_true == 1]; neg = score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    allv = np.concatenate([pos, neg])
    rank = allv.argsort().argsort().astype(float) + 1
    return (rank[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def balanced_acc(pred, yt, nclass=5):
    recs = []
    for k in range(nclass):
        m = (yt == k)
        if int(m.sum()) > 0:
            recs.append((pred[m] == k).float().mean().item())
    return float(np.mean(recs))

def sens_at_spec(y, s, spec=0.9):
    y = np.asarray(y); s = np.asarray(s)
    pos = s[y == 1]; neg = s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    thr = np.quantile(neg, spec)          # threshold giving the target specificity
    return float((pos > thr).mean())      # melanoma sensitivity at that threshold

def rule_verdict(c):
    # c: [N,7] in {0,1} -> 1 if 7-point score >= 3 (melanoma-suspicious)
    return ((c * W7).sum(1) >= 3).float()

In [ ]:
# --- 3. Policy & training (REINFORCE, multi-class, non-linear head) ---
# concept head (linear): emb -> 7 Bernoulli concepts
# diagnosis head (MLP):  concepts -> 5-class softmax. Diagnosis sees concepts only.
# Two faithfulness mechanisms are compared:
#   lam_faith  : REINFORCE reward = agreement of melanoma call with the 7pt RULE verdict (naive)
#   gamma_mono : differentiable penalty enforcing MONOTONICITY of P(mel) in the 7pt criteria (principled)
HID = 16

def init_params(seed):
    g = torch.Generator().manual_seed(seed)
    Wc = torch.zeros(7, Xtr.shape[1], requires_grad=True)
    bc = torch.zeros(7, requires_grad=True)
    W1 = (torch.randn(7, HID, generator=g) * 0.3).requires_grad_(True)
    b1 = torch.zeros(HID, requires_grad=True)
    W2 = (torch.randn(HID, 5, generator=g) * 0.3).requires_grad_(True)
    b2 = torch.zeros(5, requires_grad=True)
    return [Wc, bc, W1, b1, W2, b2]

def diag_logits(c, P):
    Wc, bc, W1, b1, W2, b2 = P
    return torch.relu(c @ W1 + b1) @ W2 + b2

def mono_penalty(base, P):
    # penalize when turning a 7pt criterion ON lowers P(melanoma); differentiable in the diag head.
    pen = 0.0
    for i in range(7):
        von = base.clone(); von[:, i] = 1.0
        voff = base.clone(); voff[:, i] = 0.0
        pon = torch.softmax(diag_logits(von, P), 1)[:, MEL_IDX]
        poff = torch.softmax(diag_logits(voff, P), 1)[:, MEL_IDX]
        pen = pen + torch.relu(poff - pon).mean()
    return pen / 7.0

def train_policy(lam_faith, mu_concept, gamma_mono=0.0, seed=0, iters=2000, lr=0.05, beta_ent=0.01):
    P = init_params(seed)
    Wc, bc = P[0], P[1]
    opt = torch.optim.Adam(P, lr=lr)
    bce = torch.nn.functional.binary_cross_entropy
    cnt = torch.tensor([(ytr == k).sum().float() for k in range(5)]).clamp(min=1)
    wcl = 1.0 / cnt; wcl = wcl / wcl.mean()        # inverse-frequency class weights (mean 1)
    idx = torch.arange(len(ytr))
    for it in range(iters):
        p = torch.sigmoid(Xtr @ Wc.T + bc).clamp(1e-5, 1 - 1e-5)
        c = torch.bernoulli(p).detach()
        logp_c = (c * torch.log(p) + (1 - c) * torch.log(1 - p)).sum(1)
        logpr = torch.log_softmax(diag_logits(c, P), 1)
        probs = logpr.exp()
        dd = torch.multinomial(probs, 1).squeeze(1).detach()
        logp_d = logpr[idx, dd]
        corr = (dd == ytr).float() * wcl[ytr]
        faith = ((dd == MEL_IDX).float() == rule_verdict(c)).float()
        r = corr + lam_faith * faith
        adv = (r - r.mean()).detach()
        ent = -(probs * logpr).sum(1)
        loss = -(adv * (logp_c + logp_d)).mean() - beta_ent * ent.mean() + mu_concept * bce(p, Ctr)
        if gamma_mono > 0:
            loss = loss + gamma_mono * mono_penalty(p.detach(), P)
        opt.zero_grad(); loss.backward(); opt.step()
    return P

def evaluate(P):
    Wc, bc = P[0], P[1]
    p = torch.sigmoid(Xte @ Wc.T + bc)
    chat = (p > 0.5).float()
    pred = diag_logits(chat, P).argmax(1)
    mel_score = torch.softmax(diag_logits(p, P), 1)[:, MEL_IDX]      # soft score for AUROC
    mel_y = (yte == MEL_IDX).numpy()
    bacc = balanced_acc(pred, yte)
    mel_auroc = auc(mel_y, mel_score.detach().numpy())
    mel_sens = sens_at_spec(mel_y, mel_score.detach().numpy(), 0.9)
    cauc = float(np.mean([auc(Cte[:, i].numpy(), p[:, i].detach().numpy()) for i in range(7)]))
    # faithfulness 1: rule-consistency of the melanoma decision under single-concept flips
    vecs = [chat]
    for i in range(7):
        f = chat.clone(); f[:, i] = 1 - f[:, i]; vecs.append(f)
    agree = []
    for v in vecs:
        mel = (diag_logits(v, P).argmax(1) == MEL_IDX).float()
        agree.append((mel == rule_verdict(v)).float().mean().item())
    # faithfulness 2: monotonicity -- turning a 7pt criterion ON must not lower P(melanoma)
    mono = []
    for i in range(7):
        v1 = chat.clone(); v1[:, i] = 1.0
        v0 = chat.clone(); v0[:, i] = 0.0
        p1 = torch.softmax(diag_logits(v1, P), 1)[:, MEL_IDX]
        p0 = torch.softmax(diag_logits(v0, P), 1)[:, MEL_IDX]
        mono.append((p1 >= p0 - 1e-6).float().mean().item())
    return {'bacc': bacc, 'mel_auroc': mel_auroc, 'mel_sens@90': mel_sens, 'concept_auroc': cauc,
            'intervention_consistency': float(np.mean(agree)), 'monotonicity': float(np.mean(mono))}

In [ ]:
# --- 4. Reference: multinomial logistic emb -> 5 classes (no concept bottleneck) ---
def reference(iters=2000, lr=0.05):
    torch.manual_seed(0)
    W = torch.zeros(Xtr.shape[1], 5, requires_grad=True)
    b = torch.zeros(5, requires_grad=True)
    opt = torch.optim.Adam([W, b], lr=lr)
    for it in range(iters):
        loss = F.cross_entropy(Xtr @ W + b, ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    logits = (Xte @ W + b).detach()
    pred = logits.argmax(1)
    mel_score = torch.softmax(logits, 1)[:, MEL_IDX]
    return {'bacc': balanced_acc(pred, yte), 'mel_auroc': auc((yte == MEL_IDX).numpy(), mel_score.numpy())}

ref = reference()
print('Reference (no bottleneck): bacc={0:.3f}  mel_auroc={1:.3f}'.format(ref['bacc'], ref['mel_auroc']))

In [ ]:
# --- 5. Compare faithfulness mechanisms (3 seeds each) ---
SEEDS = [0, 1, 2]
conditions = {
    'A_correctness_only': dict(lam_faith=0.0, mu_concept=1.0, gamma_mono=0.0),
    'B_rule_reward':      dict(lam_faith=0.5, mu_concept=1.0, gamma_mono=0.0),
    'C_monotonicity_reg': dict(lam_faith=0.0, mu_concept=1.0, gamma_mono=2.0),
}
results = {}
for name, cfg in conditions.items():
    runs = [evaluate(train_policy(seed=s, **cfg)) for s in SEEDS]
    agg = {k: float(np.mean([r[k] for r in runs])) for k in runs[0]}
    results[name] = {'mean': agg}
    print(name, {k: round(v, 3) for k, v in agg.items()})

In [ ]:
# --- 6. Verdict ---
order = ['A_correctness_only', 'B_rule_reward', 'C_monotonicity_reg']
print('condition            bacc  mel_auroc  sens@90  concept_auroc  consist  mono')
for name in order:
    m = results[name]['mean']
    print('{0:20s} {1:.3f}   {2:.3f}    {3:.3f}      {4:.3f}      {5:.3f}   {6:.3f}'.format(
        name, m['bacc'], m['mel_auroc'], m['mel_sens@90'], m['concept_auroc'],
        m['intervention_consistency'], m['monotonicity']))
print()
A = results['A_correctness_only']['mean']
B = results['B_rule_reward']['mean']
C = results['C_monotonicity_reg']['mean']
print('Naive rule reward (B) harms melanoma discrimination:',
      bool(B['mel_auroc'] < A['mel_auroc'] - 0.02))
print('Principled monotonicity (C) preserves discrimination while >= baseline faithfulness:',
      bool(C['mel_auroc'] >= A['mel_auroc'] - 0.02 and C['monotonicity'] >= A['monotonicity'] - 1e-6))

## How to read this

- **bacc** = 5-class balanced accuracy of the concept-bottleneck policy (compare to the reference, the no-bottleneck ceiling).
- **mel_auroc** = melanoma one-vs-rest AUROC.
- **concept_auroc** = how well the supervised probe recovers the 7 criteria.
- **intervention_consistency** = agreement between the melanoma decision and the 7-point rule across predicted concepts and all single-concept flips.
- **monotonicity** = fraction of (case, criterion) where turning a 7-point criterion ON does not lower P(melanoma). A non-linear head trained on accuracy alone can violate this; the faithfulness reward should restore it. This is the metric the linear binary version could not stress.

**H1 is supported if condition B raises faithfulness (consistency and/or monotonicity) while keeping mel_auroc and bacc within ~0.02 of A.** Tune `lam_faith` up if faithfulness lags, down if performance drops.

**Next (Notebook C):** cache Fitzpatrick17k + DDI with Notebook A, add Group-DRO to `train_policy`, and test whether faithfulness degrades on darker skin (H2/H3).

## How to read this

- **acc / auroc** = diagnostic performance of the concept-bottleneck policy (compare to the reference linear probe, which is the no-bottleneck ceiling).
- **concept_auroc** = how well the supervised concept probe recovers the 7 criteria (should beat the zero-shot prompts from Notebook A).
- **intervention_consistency** = how often the diagnosis head agrees with the 7-point clinical rule across the predicted concepts and all single-concept counterfactual flips. This is the faithfulness metric.

**H1 is supported if condition B raises intervention_consistency while keeping auroc within ~0.02 of A.** If accuracy drops a lot, lower `lam_faith` (e.g. 0.5) or raise `mu_concept`; if faithfulness barely moves, raise `lam_faith`.

**Next (Notebook C):** cache Fitzpatrick17k + DDI with Notebook A's code, add Group-DRO weighting to `train_policy`, and measure worst-group accuracy and whether intervention_consistency degrades on darker skin tones (H2/H3).